<a href="https://colab.research.google.com/github/rebelsuraj1506/RSNA/blob/main/solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**If you find this notebook useful, please give an upvote xD**<br>
**Thanks!**

### Inspiration and Credits 🙌
This notebook is inspired by the work of **Jason Heesang Lee**
, available at [this Kaggle project](https://www.kaggle.com/code/jasonheesanglee/rsna23-scale-h-implementation/notebook). I extend our gratitude to Jason Heesang Lee
 for sharing their insights and code.

🌟 Explore my profile and other public projects, and don't forget to share your feedback!
👉 [Visit my Profile](https://www.kaggle.com/zulqarnainali) 👈

🙏 Thank you for taking the time to review my work, and please give it a thumbs-up if you found it valuable! 👍

**Goal:** 🎯
Create a simple baseline prediction method for all test samples. This method involves using a constant prediction value derived from the mean value of each target variable, which is then scaled by proposed sample weights. The primary aim is to establish a basic predictive model.

**Sample Weights:** ⚖️
Different weights are assigned to different target labels based on their importance or significance:

- 1️⃣: All healthy labels are assigned a weight of 1.
- 2️⃣: Low-grade solid organ injuries (such as liver, spleen, and kidney) are assigned a weight of 2.
- 4️⃣: High-grade solid organ injuries are assigned a weight of 4.
- 2️⃣: Bowel injuries are assigned a weight of 2.
- 6️⃣: Extravasation is assigned a weight of 6.
- 6️⃣: The auto-generated "any_injury" label is also assigned a weight of 6.

**Evaluation and Exploration:** 🧐
Additionally, there's a method provided to assess the performance of this baseline on the training data. This evaluation can help in exploring alternative scale factors and understanding the challenges posed by unbalanced data and weighted scoring metrics.

In summary, the Weighted Mean Baseline is a simple predictive model that assigns different levels of importance (weights) to different types of target labels and uses the mean value of these labels to make predictions. It's a basic starting point for evaluating the performance of a predictive model on imbalanced data. 📊📈

## Import necessary library 🧮

In [ ]:
# Import the numpy library as 'np' for numerical operations 🧮
import numpy as np

# Import the pandas library as 'pd' for data manipulation 🐼
import pandas as pd

# Import a specific module from pandas to check data types 📊
import pandas.api.types

# Import the sklearn.metrics module for machine learning metrics 📈
import sklearn.metrics


## Load Data 📁

In [ ]:
# Load the training target data from a CSV file 📁
y_train = pd.read_csv('/kaggle/input/rsna-2023-abdominal-trauma-detection/train.csv')

# Display the first few rows of the loaded data to get a quick overview 👀
y_train.head()


## Target list 🎯

In [ ]:
# List of Targets representing various injuries and health statuses 🎯
Injuries = ['bowel_healthy', 'bowel_injury',
            'extravasation_healthy', 'extravasation_injury',
            'kidney_healthy', 'kidney_low', 'kidney_high',
            'liver_healthy', 'liver_low', 'liver_high',
            'spleen_healthy', 'spleen_low', 'spleen_high',
            'any_injury']


# Target EDA

In [ ]:
# Display summary statistics for the selected injury labels in the training target data.
# This provides an overview of the data distribution for each injury category. 📈
y_train[Injuries].describe()


## Evaluation Functions for Abdominal Trauma Detection 📊

**Explaination**:

```python
def normalize_probabilities_to_one(df: pd.DataFrame, group_columns: list) -> pd.DataFrame:
    # Normalize the sum of each row's probabilities to 100%.
    # 0.75, 0.75 => 0.5, 0.5
    # 0.1, 0.1 => 0.5, 0.5
    row_totals = df[group_columns].sum(axis=1)
    
    # Check if any row has a sum of zero, which is not allowed.
    if row_totals.min() == 0:
        raise ValueError('All rows must contain at least one non-zero prediction')
    
    # Normalize each column within the specified group columns.
    for col in group_columns:
        df[col] /= row_totals
    return df
```

Here, we have a function `normalize_probabilities_to_one` that takes a DataFrame (`df`) and a list of column names (`group_columns`) as input. It normalizes the probabilities in the DataFrame to sum to 100% for each row. Let's break down the steps:

- Calculate the sum of each row's values in the specified `group_columns`.
- Check if any row has a sum of zero, and if so, raise a `ValueError` because every row should have at least one non-zero prediction.
- Normalize each column within the `group_columns` by dividing its values by the corresponding row total.
- Return the modified DataFrame with normalized probabilities.

```python
def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    '''
    Pseudocode:
    1. For every label group (liver, bowel, etc):
        - Normalize the sum of each row's probabilities to 100%.
        - Calculate the sample weighted log loss.
    2. Derive a new any_injury label by taking the max of 1 - p(healthy) for each label group
    3. Calculate the sample weighted log loss for the new label group
    4. Return the average of all of the label group log losses as the final score.
    '''
```

This is the beginning of the `score` function. It takes three inputs: `solution`, `submission`, and `row_id_column_name`, and it returns a float value as the evaluation score. The function is outlined with pseudocode comments explaining its main steps.

Let's proceed with the next part of the function:

```python
    del solution[row_id_column_name]
    del submission[row_id_column_name]
```

These lines remove the `row_id_column_name` from both the `solution` and `submission` DataFrames. This column is not needed for scoring.

Continuing with the function:

```python
    # Basic quality checks on the inputs.
    if not pandas.api.types.is_numeric_dtype(submission.values):
        raise ValueError('All submission values must be numeric')

    if not np.isfinite(submission.values).all():
        raise ValueError('All submission values must be finite')

    if solution.min().min() < 0:
        raise ValueError('All labels must be at least zero')
    if submission.min().min() < 0:
        raise ValueError('All predictions must be at least zero')
```

Here, the code performs some basic quality checks on the `submission` DataFrame:

- Checks if all values in `submission` are numeric using `pandas.api.types.is_numeric_dtype`. Raises a `ValueError` if not.
- Checks if all values in `submission` are finite (not NaN or infinite). Raises a `ValueError` if not.
- Ensures that all labels in both `solution` and `submission` DataFrames are at least zero. Raises a `ValueError` if any label is negative.

These checks help ensure the validity of the input data.

Continuing with the function:

```python
    # Define categories for label groups.
    binary_targets = ['bowel', 'extravasation']
    triple_level_targets = ['kidney', 'liver', 'spleen']
    all_target_categories = binary_targets + triple_level_targets

    label_group_losses = []  # List to store label group log losses.

    # Loop through each label group category.
    for category in all_target_categories:
        if category in binary_targets:
            col_group = [f'{category}_healthy', f'{category}_injury']
        else:
            col_group = [f'{category}_healthy', f'{category}_low', f'{category}_high']
```

In this section, the code defines categories for label groups, specifically binary targets and triple-level targets. It also initializes an empty list, `label_group_losses`, to store the log loss for each label group.

The code then enters a loop to process each label group category. Depending on whether it's a binary target or triple-level target, it defines the `col_group` with the appropriate column names.

```python
        # Normalize probabilities to ensure they sum to 100% for each row.
        solution = normalize_probabilities_to_one(solution, col_group)

        # Check if all columns in col_group are present in the submission DataFrame.
        for col in col_group:
            if col not in submission.columns:
                raise ValueError(f'Missing submission column {col}')
        
        # Normalize probabilities in the submission DataFrame.
        submission = normalize_probabilities_to_one(submission, col_group)
```

In this part of the loop, the code does the following for each label group:

- Normalizes the probabilities in the `solution` DataFrame using the `normalize_probabilities_to_one` function, ensuring that they sum to 100% for each row.
- Checks if all columns in the `col_group` are present in the `submission` DataFrame. If any column is missing, it raises a `ValueError`.
- Normalizes probabilities in the `submission` DataFrame for the same `col_group`.

```python
        # Calculate the log loss for the current label group.
        label_group_losses.append(
            sklearn.metrics.log_loss(
                y_true=solution[col_group].values,
                y_pred=submission[col_group].values,
                sample_weight=solution[f'{category}_weight'].values
            )
        )
```

Here, the code calculates the log loss for the current label group using the `sklearn.metrics.log_loss` function. It uses the `y_true` (true labels from the `solution` DataFrame), `y_pred` (predicted probabilities from the `submission` DataFrame), and sample weights from the `solution` DataFrame.

The calculated log loss for the label group is appended to the `label_group_losses` list.

```python
    # Calculate a new any_injury label based on the max of 1 - p(healthy) for each label group.
    healthy_cols = [x + '_healthy' for x in all_target_categories]
    any_injury_labels = (1

 - solution[healthy_cols]).max(axis=1)
    any_injury_predictions = (1 - submission[healthy_cols]).max(axis=1)
```

This section calculates a new label, `any_injury`, by taking the maximum of (1 - p(healthy)) for each label group. It creates two Series: `any_injury_labels` for the true labels and `any_injury_predictions` for the predicted probabilities.

```python
    # Calculate the log loss for the any_injury label.
    any_injury_loss = sklearn.metrics.log_loss(
        y_true=any_injury_labels.values,
        y_pred=any_injury_predictions.values,
        sample_weight=solution['any_injury_weight'].values
    )
```

Here, the code calculates the log loss for the `any_injury` label using the `sklearn.metrics.log_loss` function. It uses `y_true` (true labels for `any_injury`), `y_pred` (predicted probabilities for `any_injury`), and sample weights from the `solution` DataFrame.

```python
    # Add the any_injury loss to the list of label group losses and return the mean of all losses.
    label_group_losses.append(any_injury_loss)
    return np.mean(label_group_losses)
```

The final steps involve adding the `any_injury` loss to the list of `label_group_losses`, and then returning the mean of all losses as the final evaluation score for the entire set of label groups.

This function calculates a weighted log loss score for a set of label groups based on the provided `solution` and `submission` DataFrames. The weights are used to emphasize the importance of different label groups, and the log loss measures the accuracy of the predictions. The function follows a detailed process to handle the data and compute the score. 📊🧮🏆

In [ ]:
# Define a custom exception class for potential errors in the code.
# This exception is not currently used in the code.
# However, it's good practice to define custom exceptions for future use. ⚠️
# class ParticipantVisibleError(Exception):
#     pass

# Define a function to normalize probabilities in a DataFrame to sum to 100% for each row.
# This is essential when dealing with probability predictions.
def normalize_probabilities_to_one(df: pd.DataFrame, group_columns: list) -> pd.DataFrame:
    # Normalize the sum of each row's probabilities to 100%.
    # 0.75, 0.75 => 0.5, 0.5
    # 0.1, 0.1 => 0.5, 0.5
    row_totals = df[group_columns].sum(axis=1)

    # Check if any row has a sum of zero, which is not allowed.
    if row_totals.min() == 0:
        raise ValueError('All rows must contain at least one non-zero prediction')

    # Normalize each column within the specified group columns.
    for col in group_columns:
        df[col] /= row_totals
    return df

# Define a function to calculate the evaluation score for the given solution and submission DataFrames.
# The score is based on a weighted log loss calculation.
def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    # Remove the row_id_column_name from both DataFrames as it's not needed for scoring.
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # Basic quality checks on the inputs.
    if not pandas.api.types.is_numeric_dtype(submission.values):
        raise ValueError('All submission values must be numeric')

    if not np.isfinite(submission.values).all():
        raise ValueError('All submission values must be finite')

    if solution.min().min() < 0:
        raise ValueError('All labels must be at least zero')
    if submission.min().min() < 0:
        raise ValueError('All predictions must be at least zero')

    # Define categories for label groups.
    binary_targets = ['bowel', 'extravasation']
    triple_level_targets = ['kidney', 'liver', 'spleen']
    all_target_categories = binary_targets + triple_level_targets

    label_group_losses = []  # List to store label group log losses.

    # Loop through each label group category.
    for category in all_target_categories:
        if category in binary_targets:
            col_group = [f'{category}_healthy', f'{category}_injury']
        else:
            col_group = [f'{category}_healthy', f'{category}_low', f'{category}_high']

        # Normalize probabilities to ensure they sum to 100% for each row.
        solution = normalize_probabilities_to_one(solution, col_group)

        # Check if all columns in col_group are present in the submission DataFrame.
        for col in col_group:
            if col not in submission.columns:
                raise ValueError(f'Missing submission column {col}')

        # Normalize probabilities in the submission DataFrame.
        submission = normalize_probabilities_to_one(submission, col_group)

        # Calculate the log loss for the current label group.
        label_group_losses.append(
            sklearn.metrics.log_loss(
                y_true=solution[col_group].values,
                y_pred=submission[col_group].values,
                sample_weight=solution[f'{category}_weight'].values
            )
        )

    # Calculate a new any_injury label based on the max of 1 - p(healthy) for each label group.
    healthy_cols = [x + '_healthy' for x in all_target_categories]
    any_injury_labels = (1 - solution[healthy_cols]).max(axis=1)
    any_injury_predictions = (1 - submission[healthy_cols]).max(axis=1)

    # Calculate the log loss for the any_injury label.
    any_injury_loss = sklearn.metrics.log_loss(
        y_true=any_injury_labels.values,
        y_pred=any_injury_predictions.values,
        sample_weight=solution['any_injury_weight'].values
    )

    # Add the any_injury loss to the list of label group losses and return the mean of all losses.
    label_group_losses.append(any_injury_loss)
    return np.mean(label_group_losses)


**Explaination**:

1. **Purpose of Sample Weights**: The purpose of assigning sample weights is to provide a mechanism for the evaluation function to give different levels of importance to each category of injury when calculating the overall score. It allows the evaluation to consider the significance of different types of injuries in the final assessment.

2. **Sample Weights Based on True Target**: The assignment of sample weights is based on the true target values for a given category of injury. The true target values describe the actual condition or injury status of a sample.

3. **Example Scenario**: Let's consider an example for better understanding. Suppose we are dealing with the "kidney" category of injuries. This category has three possible states: "kidney_healthy," "kidney_low," and "kidney_high."

4. **True Target Values**: If a particular sample is known to have a "low grade kidney injury," the true target values for that sample would be:

    - "kidney_healthy" = 0 (indicating that the kidney is healthy)
    - "kidney_low" = 1 (indicating a low-grade kidney injury)
    - "kidney_high" = 0 (indicating no high-grade kidney injury)

5. **Assignment of Kidney Weight**: In this scenario, the sample weight for the "kidney" category is assigned based on the true target values. Specifically, since the sample has a "low grade kidney injury" (kidney_low = 1), the kidney_weight is set to 2 for that sample.

6. **Impact on Evaluation**: This assignment of sample weights influences the evaluation process. When calculating the score for the "kidney" category, the model's predictions will be weighted by this assigned kidney_weight, reflecting the importance of correctly predicting low-grade kidney injuries in the overall evaluation.



In [ ]:
def create_training_solution(y_train):
    # Make a copy of the y_train DataFrame to avoid modifying the original data.
    sol_train = y_train.copy()

    # Assign sample weights for the 'bowel' category:
    # If 'bowel_injury' is 1 (indicating injury), set 'bowel_weight' to 2; otherwise, set it to 1 (indicating healthy).
    sol_train['bowel_weight'] = np.where(sol_train['bowel_injury'] == 1, 2, 1)

    # Assign sample weights for the 'extravasation' category:
    # If 'extravasation_injury' is 1 (indicating injury), set 'extravasation_weight' to 6; otherwise, set it to 1 (indicating healthy).
    sol_train['extravasation_weight'] = np.where(sol_train['extravasation_injury'] == 1, 6, 1)

    # Assign sample weights for the 'kidney' category:
    # If 'kidney_low' is 1 (indicating low-grade injury), set 'kidney_weight' to 2;
    # If 'kidney_high' is 1 (indicating high-grade injury), set 'kidney_weight' to 4;
    # Otherwise, set 'kidney_weight' to 1 (indicating healthy).
    sol_train['kidney_weight'] = np.where(sol_train['kidney_low'] == 1, 2, np.where(sol_train['kidney_high'] == 1, 4, 1))

    # Assign sample weights for the 'liver' category (similar to 'kidney').
    sol_train['liver_weight'] = np.where(sol_train['liver_low'] == 1, 2, np.where(sol_train['liver_high'] == 1, 4, 1))

    # Assign sample weights for the 'spleen' category (similar to 'kidney' and 'liver').
    sol_train['spleen_weight'] = np.where(sol_train['spleen_low'] == 1, 2, np.where(sol_train['spleen_high'] == 1, 4, 1))

    # Assign sample weights for the 'any_injury' category:
    # If 'any_injury' is 1 (indicating injury), set 'any_injury_weight' to 6; otherwise, set it to 1 (indicating healthy).
    sol_train['any_injury_weight'] = np.where(sol_train['any_injury'] == 1, 6, 1)

    # Return the DataFrame with sample weights assigned to each category.
    return sol_train


**Explaination**:

```python
# Use the create_training_solution function to assign sample weights to injury categories.
solution_train = create_training_solution(y_train)
```

Here, `create_training_solution(y_train)` is called to create a DataFrame `solution_train` with the assigned sample weights for different injury categories. This DataFrame will be used for scoring.

```python
# Make a constant prediction using the mean of the training data for all injury categories.
y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()
```

This part creates a constant prediction `y_pred` for all injury categories. It copies the `y_train` DataFrame and replaces the predictions for all injury categories (`Injuries`) with the mean value of each category calculated from the training data. This essentially predicts a constant value for all samples.

```python
# Calculate the score without any scaling using the score function.
no_scale_score = score(solution_train, y_pred, 'patient_id')
```

Here, the `score` function is called to evaluate the model's performance using `solution_train` as the ground truth and `y_pred` as the predicted values. The `'patient_id'` column is used as the row identifier.



In [ ]:
solution_train = create_training_solution(y_train)

# predict a constant using the mean of the training data
y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()

no_scale_score = score(solution_train,y_pred,'patient_id')
print(f'Training score without scaling: {no_scale_score}')

## Weight Scaling and Score Calculation 🧮🏋️

In [ ]:
# Cell Title: Automated Weight Scaling Optimization 🔄📊

# Define the categories to be scaled by different factors.
scale_by_2 = ['liver_low', 'spleen_low', 'kidney_low', 'spleen_high']  # Categories to be scaled by 2
scale_by_4 = ['bowel_injury', 'kidney_high', 'liver_high']  # Categories to be scaled by 4
scale_by_6 = ['extravasation_injury']  # Categories to be scaled by 6
scale_healthy = ['kidney_healthy', 'bowel_healthy']  # Categories to be scaled by a healthy factor

# Define a range of values to try for the MULTIPLIER (scale factor for all categories).
multiplier_values = [0.98, 0.985, 0.99, 0.995]

# Initialize variables to keep track of the best scale factors and score.
best_sf_2 = best_sf_4 = best_sf_6 = best_scale_h = None
best_score = float('inf')  # Initialize with a high value (you can also use -1 if lower scores are better)

# Iterate through each multiplier value and find the best combination.
for MULTIPLIER in multiplier_values:
    # Define specific scale factors for each category using the current MULTIPLIER.
    sf_2 = 2.80 * MULTIPLIER  # Scale factor for categories to be scaled by 2
    sf_4 = 6.90 * MULTIPLIER  # Scale factor for categories to be scaled by 4
    sf_6 = 28.99 * MULTIPLIER  # Scale factor for categories to be scaled by 6
    scale_h = 0.99815 * MULTIPLIER  # Scale factor for healthy categories

    # Prepare the solution_train DataFrame with assigned sample weights.
    solution_train = create_training_solution(y_train)

    # Create initial predictions by setting them to the mean values.
    y_pred = y_train.copy()
    y_pred[Injuries] = y_train[Injuries].mean().tolist()

    # Scale the predictions for different categories by their respective factors.
    y_pred[scale_by_2] *= sf_2
    y_pred[scale_by_4] *= sf_4
    y_pred[scale_by_6] *= sf_6
    y_pred[scale_healthy] *= scale_h

    # Calculate the score after weight scaling.
    weight_scale_score = score(solution_train, y_pred, 'patient_id')

    # Check if the current combination of scale factors results in a better score.
    if weight_scale_score < best_score:
        best_score = weight_scale_score
        best_sf_2, best_sf_4, best_sf_6, best_scale_h = sf_2, sf_4, sf_6, scale_h

# Print the best scale factors and the corresponding best score.
print(f'Best Scale Factors (sf_2, sf_4, sf_6, scale_h): ({best_sf_2}, {best_sf_4}, {best_sf_6}, {best_scale_h})')
print(f'Best Training Score with Optimized Scaling: {best_score}')


## Improved Scaling for Better Score 📈🚀

In [ ]:
# Updated scale factors to potentially improve the score
#sf_2 = 2
#sf_4 = 4
#sf_6 = 14

# Prepare the solution_train DataFrame with assigned sample weights
solution_train = create_training_solution(y_train)

# Reset the predictions
y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()

# Scale each target category with the updated scale factors
y_pred[scale_by_2] *= sf_2
y_pred[scale_by_4] *= sf_4
y_pred[scale_by_6] *= sf_6
y_pred[scale_healthy] *= scale_h

# Calculate the score after better scaling
improved_scale_score = score(solution_train, y_pred, 'patient_id')

# Print the training score with better scaling
print(f'Training score with better scaling: {improved_scale_score}')


## Submission Preparation and Scaling 📤🔍

In [ ]:
# Load the submission template from a CSV file
submission = pd.read_csv('/kaggle/input/rsna-2023-abdominal-trauma-detection/sample_submission.csv')

# Set the output for injury categories to the mean of the training data
submission[Injuries] = y_train[Injuries].mean().tolist()

# Scale each category in the submission DataFrame by the desired scale factors
submission[scale_by_2] *= sf_2
submission[scale_by_4] *= sf_4
submission[scale_by_6] *= sf_6
submission[scale_healthy] *= scale_h


# Save the prepared submission to a CSV file



In [ ]:
submission

In [ ]:
submission[["bowel_healthy"]] += 0.07
submission[["extravasation_healthy"]] += 0.15
submission[["kidney_healthy"]] += 0.07
submission[["liver_healthy"]] += 0.07
submission[["spleen_healthy"]] += 0.07

submission[["extravasation_injury"]] -= 0.08
submission[["bowel_injury"]] += 0.02

submission[["kidney_low"]] += 0.09
submission[["liver_low"]] -= 0.03
# submission[["spleen_low"]] -= 0.03

submission[["kidney_high"]] += 0.03

submission.to_csv('submission.csv', index=False)

## Explore More! 👀
Thank you for exploring this notebook! If you found this notebook insightful or if it helped you in any way, I invite you to explore more of my work on my profile.

👉 [Visit my Profile](https://www.kaggle.com/zulqarnainali) 👈

## Feedback and Gratitude 🙏
We value your feedback! Your insights and suggestions are essential for our continuous improvement. If you have any comments, questions, or ideas to share, please don't hesitate to reach out.

📬 Contact me via email: [zulqar445ali@gmail.com](mailto:zulqar445ali@gmail.com)

I would like to express our heartfelt gratitude for your time and engagement. Your support motivates us to create more valuable content.

Happy coding and best of luck in your data science endeavors! 🚀
